### Create Data

In [0]:
data = [
    (1, "A", 5000),
    (2, "B", 6000),
    (3, "C", 7000)
]

columns = ["id", "name", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

### Write as Delta Table

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("/Volumes/workspace/default/day6")

### Read Delta Table

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/day6")
df.display()

### INSERT

In [0]:
new_data = [(4, "D", 8000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/workspace/default/day6")

### UPDATE

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/workspace/default/day6")

delta_table.update(
    condition = "id = 2",
    set = {"salary": "6500"}
)

### DELETE

In [0]:
delta_table.delete("id = 1")

### MERGE INTO

### How do we handle incremental load with updates?”

### Existing Data

In [0]:
target_df = spark.read.format("delta").load("/Volumes/workspace/default/day6")
target_df.display()

### New Data

In [0]:
updates = [
    (2, "B", 7000),  # update
    (5, "E", 9000)   # insert
]

updates_df = spark.createDataFrame(updates, columns)
updates_df.display()

### MERGE

In [0]:
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "salary": "source.salary"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "name": "source.name",
    "salary": "source.salary"
}).execute()

### If id exists → UPDATE
### If id not exists → INSERT
### This is UPSERT

### Schema Evolution

### Problem:

### New column arrives

### Example:

In [0]:
new_data = [(6, "F", 10000, "India")]
new_columns = ["id", "name", "salary", "country"]

df_new = spark.createDataFrame(new_data, new_columns)
df_new.display()

### Enable schema evolution:

In [0]:
df_new.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/day6")

In [0]:
df_read=spark.read.format('delta').load("/Volumes/workspace/default/day6")
df_read.display()


### Time Travel

### View History

In [0]:
delta_table.history().display()

### Read Old Version

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/day6")
df_old.display()

### Restore Table

In [0]:
delta_table.restoreToVersion(0)